# 02 — Federated Partitioning (Non-IID Geographic Split)

Visualise London / Manchester / Birmingham client distributions for the PP-XAI dissertation.

**Inputs:** `results/tables/partition_stats.csv`, `data/federated/client_*.csv`  
**Outputs:** figures under `results/figures/` (including `fig8_client_distributions.png`)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..") if not Path("data/federated").exists() else Path(".")
if not (ROOT / "data" / "federated" / "client_london.csv").exists():
    ROOT = Path(".")

fig_dir = ROOT / "results" / "figures"
table_dir = ROOT / "results" / "tables"
fig_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT.resolve())


## Load partition stats and client CSVs


In [ ]:
clients = ["london", "manchester", "birmingham"]
frames = []
for city in clients:
    path = ROOT / "data" / "federated" / f"client_{city}.csv"
    part = pd.read_csv(path)
    part["client"] = city
    frames.append(part)
    print(f"{city}: n={len(part)}, mean_energy={part['energy_consumption'].mean():.2f}")

df = pd.concat(frames, ignore_index=True)
stats = pd.read_csv(table_dir / "partition_stats.csv")
print("\npartition_stats.csv:")
display(stats)
print("Combined shape:", df.shape)


## Detailed non-IID statistics


In [ ]:
detailed = (
    df.groupby("client")
    .agg(
        records=("energy_consumption", "count"),
        mean_energy=("energy_consumption", "mean"),
        std_energy=("energy_consumption", "std"),
        median_energy=("energy_consumption", "median"),
        mean_floor_area=("floor_area", "mean"),
        pct_commercial=("building_typology", lambda s: 100.0 * (s == "commercial").mean()),
    )
    .reset_index()
)
detailed.to_csv(table_dir / "partition_detailed_stats.csv", index=False)
display(detailed)

overall_mean = df["energy_consumption"].mean()
max_rel = ((stats["mean_energy"] - overall_mean).abs() / overall_mean * 100).max()
print(f"Overall mean energy: {overall_mean:.2f}")
print(f"Max |client mean − overall| / overall: {max_rel:.2f}% (mild label skew)")


## Client sizes and mean energy


In [ ]:
order = ["london", "manchester", "birmingham"]

plt.figure(figsize=(6, 4))
sns.barplot(data=stats, x="client", y="records", color="#4C72B0", order=order)
plt.title("Federated client sizes (geographic partition)")
plt.ylabel("Number of records")
plt.xlabel("Client (city)")
plt.tight_layout()
plt.savefig(fig_dir / "partition_client_sizes.png", dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
sns.barplot(data=stats, x="client", y="mean_energy", color="#55A868", order=order)
plt.title("Mean energy consumption by client (mild non-IID)")
plt.ylabel("Mean energy consumption")
plt.xlabel("Client (city)")
plt.tight_layout()
plt.savefig(fig_dir / "partition_mean_energy.png", dpi=150)
plt.show()


## Energy distribution boxplot (Fig. 8)


In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x="client", y="energy_consumption", order=order)
plt.title("Client energy distributions (non-IID geographic split)")
plt.ylabel("Energy consumption")
plt.xlabel("Federated client")
plt.tight_layout()
plt.savefig(fig_dir / "fig8_client_distributions.png", dpi=150)
plt.savefig(fig_dir / "partition_energy_boxplot.png", dpi=150)
plt.show()


## Typology mix by client


In [ ]:
mix = df.groupby(["client", "building_typology"]).size().reset_index(name="count")
plt.figure(figsize=(7, 4))
sns.barplot(data=mix, x="client", y="count", hue="building_typology", order=order)
plt.title("Building typology mix by federated client")
plt.ylabel("Count")
plt.xlabel("Client (city)")
plt.tight_layout()
plt.savefig(fig_dir / "partition_typology_mix.png", dpi=150)
plt.show()

print("Saved figures:")
for name in [
    "partition_client_sizes.png",
    "partition_mean_energy.png",
    "fig8_client_distributions.png",
    "partition_energy_boxplot.png",
    "partition_typology_mix.png",
]:
    print(" -", fig_dir / name)
